# Phase 4: Gemma 4 — Multimodal Fine-Tuning (Image + FIM Text)

This stage trains on IDE screenshot + code pairs so the model can leverage visual context.

- Uses stable HuggingFace stack (`transformers` + `peft` + `bitsandbytes`)
- Avoids `unsloth` runtime conflicts seen in Phase 2
- Prefers `E2B` on Kaggle T4 for reliability; can be switched to `E4B` later

**Papers referenced:**
- [VisCodex] Unified Multimodal Code Generation
- [VinciCoder] Multimodal Code via Visual RL
- [Design2Code] Benchmarking Screenshot-to-Code

**Prerequisites:**
1. Phase 2 checkpoint uploaded to Kaggle input (optional warm-start)
2. Phase 3 multimodal dataset uploaded (expected fields: `image`, `text`, `language`)

In [ ]:
# Cell 1: Install dependencies (stable stack)
!pip install -q --upgrade transformers 2>&1 | tail -3
!pip install -q peft bitsandbytes accelerate trl datasets pyyaml tqdm Pillow 2>&1 | tail -3

import torch, transformers, peft, bitsandbytes
print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda}")
print(f"transformers {transformers.__version__} | peft {peft.__version__} | bnb {bitsandbytes.__version__}")
print(f"Visible GPUs: {torch.cuda.device_count()}")
assert torch.cuda.device_count() >= 1, "GPU is required"

In [ ]:
# Cell 2: Utilities
import gc
import torch

def used_cuda_devices(model):
    dmap = getattr(model, "hf_device_map", None)
    used = set()
    if isinstance(dmap, dict):
        for v in dmap.values():
            if isinstance(v, int):
                used.add(v)
            elif isinstance(v, str) and v.startswith("cuda:"):
                used.add(int(v.split(":", 1)[1]))
    if not used:
        for _, p in model.named_parameters():
            if p.device.type == "cuda":
                used.add(0 if p.device.index is None else p.device.index)
                if len(used) >= 2:
                    break
    return sorted(used)

print("Utility helpers ready")

In [ ]:
# Cell 3: Load Gemma 4 model + processor (optionally warm-start from Phase 2 adapter)
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel

MODEL_CANDIDATES = ["google/gemma-4-E2B", "google/gemma-4-E4B"]
PHASE2_ADAPTER_CANDIDATES = [
    "/kaggle/input/models/lasdwop/gemma4-tab/pytorch/default/1",
    "/kaggle/input/gemma4-phase2/gemma4-e4b-fim-final",
    "/kaggle/input/gemma4-phase2/gemma-4-e2b-fim-final",
    "/kaggle/input/gemma4-phase2",
    "/kaggle/working/gemma4-e4b-fim-final",
]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = None
processor = None
tokenizer = None
MODEL_NAME = None

for candidate in MODEL_CANDIDATES:
    try:
        print(f"Trying base model: {candidate}")
        proc = AutoProcessor.from_pretrained(candidate, trust_remote_code=True)
        mdl = AutoModelForCausalLM.from_pretrained(
            candidate,
            quantization_config=bnb_config,
            device_map="balanced",
            max_memory={i: "14GiB" for i in range(torch.cuda.device_count())},
            dtype=torch.float16,
            attn_implementation="sdpa",
            trust_remote_code=True,
            low_cpu_mem_usage=True,
        )
        MODEL_NAME = candidate
        processor = proc
        model = mdl
        break
    except Exception as e:
        print(f"Failed {candidate}: {type(e).__name__}: {e}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

if model is None:
    raise RuntimeError("Could not load Gemma 4 base model")

# Optional warm-start from Phase 2 LoRA adapter.
adapter_loaded = False
for p in PHASE2_ADAPTER_CANDIDATES:
    p = Path(p)
    if (p / "adapter_config.json").exists() and (p / "adapter_model.safetensors").exists():
        print(f"Loading Phase 2 adapter from: {p}")
        model = PeftModel.from_pretrained(model, str(p), is_trainable=False)
        model = model.merge_and_unload()
        adapter_loaded = True
        break

print(f"Model loaded: {MODEL_NAME}")
print(f"Phase 2 adapter warm-start: {adapter_loaded}")

# Attach fresh multimodal LoRA adapters.
for p in model.parameters():
    p.requires_grad = False
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()
if hasattr(model, "enable_input_require_grads"):
    model.enable_input_require_grads()

wanted_suffixes = {
    "q_proj.linear", "k_proj.linear", "v_proj.linear", "o_proj.linear",
    "gate_proj.linear", "up_proj.linear", "down_proj.linear",
}
module_names = [n for n, _ in model.named_modules()]
target_modules = sorted({s for s in wanted_suffixes if any(n.endswith(s) for n in module_names)})
if not target_modules:
    raise RuntimeError("No supported target modules found for LoRA")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=target_modules,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

tokenizer = getattr(processor, "tokenizer", None)
if tokenizer is None:
    raise RuntimeError("Processor has no tokenizer")

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.4f}%)")
print(f"Used GPUs: {used_cuda_devices(model)}")

In [ ]:
# Cell 4: Load multimodal dataset (image + text)
from pathlib import Path
from datasets import load_from_disk

MM_DATASET_CANDIDATES = [
    "/kaggle/input/datasets/lasdwop/multimodal-dataset",
    "/kaggle/input/gemma4-multimodal-dataset",
    "/kaggle/input/gemma4-multimodal-dataset/multimodal_dataset",
    "/kaggle/input/multimodal-dataset",
]

mm_ds = None
for p in MM_DATASET_CANDIDATES:
    pth = Path(p)
    if not pth.exists():
        continue
    try:
        ds = load_from_disk(str(pth))
        mm_ds = ds["train"] if hasattr(ds, "keys") and "train" in ds else ds
        print(f"Loaded multimodal dataset from: {p} | samples: {len(mm_ds)}")
        break
    except Exception as e:
        print(f"Found {p} but could not load: {e}")

if mm_ds is None:
    raise RuntimeError("Multimodal dataset not found. Upload Phase 3 dataset to Kaggle input.")

# Bound training for stability on free GPUs.
MAX_MM_SAMPLES = 10000
if len(mm_ds) > MAX_MM_SAMPLES:
    mm_ds = mm_ds.select(range(MAX_MM_SAMPLES))

# Clip text later in the collator (avoids writing temp/cache files to read-only /kaggle/input).
MAX_TEXT_CHARS = 500

# IMPORTANT: avoid dataset.shuffle() on /kaggle/input (read-only).
# Trainer's RandomSampler already shuffles each epoch.
train_dataset = mm_ds
print(f"Using multimodal training samples: {len(train_dataset)} | max_chars={MAX_TEXT_CHARS}")

In [ ]:
# Cell 5: Train multimodal model (image + text collator)
from transformers import TrainingArguments, Trainer
import torch

OUTPUT_DIR = "/kaggle/working/gemma4-multimodal"
TRAIN_MAX_SEQ_LENGTH = 192

# Keep Trainer single-process; model itself can still be layer-sharded.
model.is_parallelizable = True
model.model_parallel = True

def multimodal_collate(batch):
    import io
    from PIL import Image

    encoded = []

    for ex in batch:
        img = ex.get("image")
        txt = ex.get("text", "")

        # Normalize dataset decoding quirks.
        if isinstance(img, (list, tuple)):
            img = img[0] if len(img) > 0 else None
        if isinstance(txt, (list, tuple)):
            txt = txt[0] if len(txt) > 0 else ""

        if isinstance(img, dict):
            if img.get("bytes") is not None:
                img = Image.open(io.BytesIO(img["bytes"]))
            elif img.get("path"):
                img = Image.open(img["path"])
            else:
                img = None

        if img is None:
            continue

        if hasattr(img, "convert"):
            img = img.convert("RGB")

        txt = str(txt)[:MAX_TEXT_CHARS]

        # Encode one sample at a time to guarantee 1 image <-> 1 text pairing.
        item = processor(
            text=[txt],
            images=[img],
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=TRAIN_MAX_SEQ_LENGTH,
        )
        encoded.append(item)

    if len(encoded) == 0:
        raise RuntimeError("Collator got an empty/invalid multimodal batch")

    batch_inputs = {}
    keys = encoded[0].keys()
    for k in keys:
        batch_inputs[k] = torch.cat([e[k] for e in encoded], dim=0)

    labels = batch_inputs["input_ids"].clone()
    pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0
    labels[labels == pad_id] = -100
    batch_inputs["labels"] = labels
    return batch_inputs

train_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=8e-5,
    fp16=False,
    bf16=False,
    optim="adamw_torch",
    warmup_steps=30,
    lr_scheduler_type="cosine",
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=train_args,
    train_dataset=train_dataset,
    data_collator=multimodal_collate,
)
trainer.args._n_gpu = 1

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n_trainable:,}")
print(f"Training samples: {len(train_dataset)}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Starting multimodal training...")
trainer.train()
print("Multimodal training complete!")

In [ ]:
# Cell 6: Save Phase 4 outputs
import shutil
from pathlib import Path
from IPython.display import FileLink, display

model_tag = MODEL_NAME.split("/")[-1].lower()
SAVE_PATH = Path(f"/kaggle/working/{model_tag}-multimodal-lora-final")

# For PEFT models this saves adapter weights; for full models it saves full checkpoint.
model.save_pretrained(str(SAVE_PATH))
processor.save_pretrained(str(SAVE_PATH))
print(f"Multimodal checkpoint saved to {SAVE_PATH}")

# Optional: merge LoRA for a standalone HF checkpoint (can be slower and larger).
MERGED_PATH = Path(f"/kaggle/working/{model_tag}-multimodal-merged")
try:
    merged_model = model.merge_and_unload()
    merged_model.save_pretrained(str(MERGED_PATH))
    processor.save_pretrained(str(MERGED_PATH))
    print(f"Merged checkpoint saved to {MERGED_PATH}")
except Exception as e:
    print(f"Skipping merged save: {e}")

# Create zip for easy download.
zip_base = Path(f"/kaggle/working/{SAVE_PATH.name}")
zip_path = Path(shutil.make_archive(str(zip_base), "zip", root_dir=str(SAVE_PATH)))
print(f"Created zip: {zip_path}")
print("Download link:")
display(FileLink(str(zip_path)))